# Tokenization与Attention：从字符合并到信息读取

目标：看到BPE学到的合并规则、手算QK分数与V汇总、缩放对方差的影响、因果mask和RoPE相对位置。教学输入与随机权重，不是训练好的语言模型。

[Tokenizer正文](../01-concepts/tokenization/README.md) · [Transformer正文](../01-concepts/transformer/README.md) · [源码](../05-code/model_mechanics.py)。

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
here = Path.cwd().resolve()
repo = next(p for p in [here, *here.parents] if (p / '10-Knowledge').is_dir())
sys.path.insert(0, str(repo / '10-Knowledge' / '02-foundation-models' / '05-code'))
from model_mechanics import *
np.set_printoptions(precision=6, suppress=True)
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('Data: synthetic teaching examples; no model download or API call')

Python: 3.12.13 NumPy: 2.5.2
Data: synthetic teaching examples; no model download or API call


## 1. BPE训练规则与编码分开

字符级实现按空格分词，添加词尾符号。它不能用于估算商用模型token数，也没有包含byte-level BPE与Unicode规范化。

In [2]:
corpus='low low low lower lower newest newest widest'
rules=train_bpe(corpus,merges=8)
print('learned merge rules:')
for i,pair in enumerate(rules,1):print(i,pair)
for word in ['low','lower','lowest','新词']:
    tokens=bpe_encode(word,rules)
    restored=''.join(tokens).replace('</w>','')
    print(word,'->',tokens,'->',restored)
    assert restored==word
assert rules==train_bpe(corpus,merges=8)

learned merge rules:
1 ('l', 'o')
2 ('lo', 'w')
3 ('e', 's')
4 ('es', 't')
5 ('est', '</w>')
6 ('low', '</w>')
7 ('e', 'r')
8 ('e', 'w')
low -> ['low</w>'] -> low
lower -> ['low', 'er', '</w>'] -> lower
lowest -> ['low', 'est</w>'] -> lowest
新词 -> ['新', '词', '</w>'] -> 新词


## 2. 同一组Q/K，修改V只改变读取内容

q=[√2,0]，K的两行为[1,0]和[0,1]，缩放分数应为[1,0]。权重约为[0.731,0.269]；V提供要汇总的内容。

In [3]:
q=np.array([[np.sqrt(2),0.]])
k=np.eye(2)
v=np.array([[10.,0.],[0.,20.]])
out,weights=attention(q,k,v)
print('raw scores:',q@k.T)
print('scaled scores:',q@k.T/np.sqrt(2))
print('weights:',weights,'output:',out)
changed_v=v.copy();changed_v[1,1]=200
changed_out,unchanged_weights=attention(q,k,changed_v)
print('changed V -> output:',changed_out)
assert np.allclose(weights,unchanged_weights)
assert np.allclose(out,[[7.3105857863,5.3788284274]])
assert np.isclose(changed_out[0,1],out[0,1]*10)

raw scores: [[1.414214 0.      ]]
scaled scores: [[1. 0.]]
weights: [[0.731059 0.268941]] output: [[7.310586 5.378828]]
changed V -> output: [[ 7.310586 53.788284]]


## 3. 为什么缩放是√dk而非dk

独立标准正态分量满足推导假设。用8000对随机向量估计方差；原始点积方差接近dk，除√dk后接近1。真实训练后独立性与单位方差未必成立。

In [4]:
rng=np.random.default_rng(7)
for dimension in [4,16,64,256]:
    q_random=rng.normal(size=(8000,dimension))
    k_random=rng.normal(size=(8000,dimension))
    raw=np.sum(q_random*k_random,axis=-1)
    scaled=raw/np.sqrt(dimension)
    print('dk:',dimension,'raw variance:',round(float(raw.var()),3),'scaled variance:',round(float(scaled.var()),3))
    assert .9<scaled.var()<1.1
for scores in [[0.,2.],[0.,20.]]:
    a=stable_softmax(scores)
    print('scores:',scores,'weights:',a,'diagonal softmax derivatives:',a*(1-a))

dk: 4 raw variance: 3.98 scaled variance: 0.995
dk: 16 raw variance: 15.725 scaled variance: 0.983
dk: 64 raw variance: 63.691 scaled variance: 0.995
dk: 256 raw variance: 252.833 scaled variance: 0.988
scores: [0.0, 2.0] weights: [0.119203 0.880797] diagonal softmax derivatives: [0.104994 0.104994]
scores: [0.0, 20.0] weights: [0. 1.] diagonal softmax derivatives: [0. 0.]


## 4. 因果mask要经得住反例

对随机输入运行因果注意力，再改变最后一个token的K/V。前三个输出必须不变；没有mask时通常会变化。并检查全屏蔽行明确报错。

In [5]:
rng=np.random.default_rng(9)
x=rng.normal(size=(4,3))
allowed=np.tril(np.ones((4,4),bool))
masked,_=attention(x,x,x,allowed)
new=x.copy();new[-1]+=100
masked_changed,_=attention(new,new,new,allowed)
plain,_=attention(x,x,x)
plain_changed,_=attention(new,new,new)
print('masked previous-position max difference:',np.max(abs(masked[:-1]-masked_changed[:-1])))
print('unmasked previous-position max difference:',np.max(abs(plain[:-1]-plain_changed[:-1])))
assert np.allclose(masked[:-1],masked_changed[:-1])
assert not np.allclose(plain[:-1],plain_changed[:-1])
try:attention(q,k,v,np.zeros((1,2),bool))
except ValueError as error:print('empty attention row rejected:',error)
else:raise AssertionError('Empty row accepted')

masked previous-position max difference: 0.0
unmasked previous-position max difference: 102.07280553119236
empty attention row rejected: Every softmax row needs at least one finite logit


## 5. RoPE的二维旋转依赖相对位移

验证(R(mθ)q)ᵀ(R(nθ)k)=qᵀR((n-m)θ)k。这里只用一对维度，真实模型有多对不同频率维度。

In [6]:
def rotation(angle):
    c,s=np.cos(angle),np.sin(angle)
    return np.array([[c,-s],[s,c]])
q2=np.array([1.,2.]);k2=np.array([3.,-1.]);theta=.2;m,n=4,9
left=(rotation(m*theta)@q2)@(rotation(n*theta)@k2)
right=q2@(rotation((n-m)*theta)@k2)
shifted=(rotation((m+100)*theta)@q2)@(rotation((n+100)*theta)@k2)
print('direct:',left,'relative:',right,'common shifted:',shifted)
assert np.allclose([left,shifted],right)

direct: 6.4305991995234155 relative: 6.4305991995234155 common shifted: 6.4305991995234155


## 观察与边界

BPE规则固定后可编码新词；Attention权重由Q/K决定，V决定被汇总的信息；缩放、mask和RoPE分别有数值反例或等价检查。随机Q/K没有语义，这些实验不证明“哪个词应当关注哪个词”。下一步做[解码与缓存实验](02-decoding-cache-and-precision.ipynb)，检验逐步计算是否与完整因果计算一致。